System architecture

```
kaggle database
        │
        ▼
[ ingestion_agent ]   ← loads the data (reagan)
        │
        ▼
[ preprocess_agent ]    ← GPT-4o-mini decides how to handle missing values (reagan)
No missing values. Do one-hot encoding
        │
        ▼
[ model_1_training_agent ]    ← trains model 1: Random Forest. GPT-4o-mini writes the codes (aish)
[ model_2_training_agent ]    ← trains model 2: XGBoost(aish)
        │
        ▼
[ model_1_evaluation_agent ]  ← GPT-4o-mini interprets the results in plain English (shaorong)
[ model_2_evaluation_agent ]  ← GPT-4o-mini interprets the results in plain English (shaorong)
        │
        ▼
[ model_selection_agent ]  ← GPT-4o-mini compares the model results and select the better model with a reasoning (yangyi)
        │
        ▼
[ run_inference_agent ]  ← GPT-4o-mini reads user inputs and returns prediction based on the chosen model (yangyi)
        │
        ▼
[ recommendation_agent ]  ←  GPT-4o-mini provides business insight and recommendations to user
```

# Set up, data ingestion

In [23]:
!pip install langgraph langchain-openai langchain-core python-frontmatter openai pandas scikit-learn pydantic numpy -q kagglehub python-dotenv xgboost
import json
import pandas as pd
import numpy as np

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohankrishnathalla/global-ai-and-data-jobs-salary-dataset")

print("Path to dataset files:", path)

/home/azureuser/5151_project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/azureuser/.cache/kagglehub/datasets/mohankrishnathalla/global-ai-and-data-jobs-salary-dataset/versions/1


In [3]:
import os
filename = os.listdir(path)[0]
print(filename)

global_ai_jobs.csv


In [4]:
import pandas as pd
job_df = pd.read_csv(os.path.join(path, filename))
print(f'Loaded {filename} with {len(job_df)} samples.')

Loaded global_ai_jobs.csv with 90000 samples.


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
# from google.colab import userdata

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI()

print("Ready. OpenAI client created.")

Ready. OpenAI client created.


In [6]:
job_df.describe()

,id,experience_years,salary_usd,bonus_usd,interview_rounds,year,weekly_hours,company_rating,job_openings,hiring_difficulty_score,...,vacation_days,skill_demand_score,automation_risk,job_security_score,career_growth_score,work_life_balance_score,promotion_speed,salary_percentile,cost_of_living_index,employee_satisfaction
count,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,...,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000,90000.000000
mean,45000.500000,7.028133,96546.249222,13028.418722,4.495689,2023.003200,45.476268,3.998004,17.521867,55.028604,...,19.986367,50.461200,50.357544,75.563533,57.198544,69.146478,38.439633,50.542411,1.503042,72.733100
std,25980.906451,5.889327,43935.479553,7886.738085,1.704553,2.002624,5.475497,0.461914,7.848576,17.901451,...,6.069607,28.853798,28.845671,11.316485,12.900225,13.213996,18.429221,28.891570,0.576449,8.124018
min,1.000000,0.000000,28000.000000,1404.000000,2.000000,2020.000000,36.000000,3.200000,1.000000,0.000000,...,10.000000,1.000000,1.000000,29.000000,25.000000,25.000000,12.000000,1.000000,0.500000,42.000000
25%,22500.750000,2.000000,64676.750000,7104.750000,3.000000,2021.000000,40.700000,3.600000,12.000000,42.881134,...,15.000000,25.000000,25.000000,68.000000,48.000000,59.000000,24.000000,25.000000,1.010000,67.000000
50%,45000.500000,6.000000,87544.000000,11279.000000,4.000000,2023.000000,45.500000,4.000000,17.000000,55.066089,...,20.000000,51.000000,50.000000,77.000000,57.000000,69.000000,37.000000,51.000000,1.510000,73.000000
75%,67500.250000,12.000000,123906.000000,16997.250000,6.000000,2025.000000,50.200000,4.400000,23.000000,67.118119,...,25.000000,75.000000,75.000000,84.000000,66.000000,79.000000,51.000000,76.000000,2.000000,78.000000
max,90000.000000,19.000000,300622.000000,57681.000000,7.000000,2026.000000,55.000000,4.800000,50.000000,100.000000,...,30.000000,100.000000,100.000000,99.000000,99.000000,98.000000,98.000000,100.000000,2.500000,99.000000


In [7]:
job_df.dtypes

id                           int64
country                        str
job_role                       str
ai_specialization              str
experience_level               str
experience_years             int64
salary_usd                   int64
bonus_usd                    int64
education_required             str
industry                       str
company_size                   str
interview_rounds             int64
year                         int64
work_mode                      str
weekly_hours               float64
company_rating             float64
job_openings                 int64
hiring_difficulty_score    float64
layoff_risk                float64
ai_adoption_score            int64
company_funding_billion    float64
economic_index             float64
ai_maturity_years            int64
offer_acceptance_rate      float64
tax_rate_percent           float64
vacation_days                int64
skill_demand_score           int64
automation_risk              int64
job_security_score  

In [8]:
job_df.isnull().sum()

id                         0
country                    0
job_role                   0
ai_specialization          0
experience_level           0
experience_years           0
salary_usd                 0
bonus_usd                  0
education_required         0
industry                   0
company_size               0
interview_rounds           0
year                       0
work_mode                  0
weekly_hours               0
company_rating             0
job_openings               0
hiring_difficulty_score    0
layoff_risk                0
ai_adoption_score          0
company_funding_billion    0
economic_index             0
ai_maturity_years          0
offer_acceptance_rate      0
tax_rate_percent           0
vacation_days              0
skill_demand_score         0
automation_risk            0
job_security_score         0
career_growth_score        0
work_life_balance_score    0
promotion_speed            0
salary_percentile          0
cost_of_living_index       0
employee_satis

# Create shared states

In [9]:
from typing import Any, Optional
from pydantic import BaseModel, Field

os.makedirs("models", exist_ok=True)

class ModelAgentState(BaseModel):
    # Pipeline log
    messages: list[str] = Field(default_factory=list)

    # Required fields
    raw_df: Optional[Any] = job_df
    processed_df: Optional[Any] = None
    random_forest_model: Optional[Any] = None
    xgboost_model: Optional[Any] = None
    X_train: Optional[Any] = None
    y_train: Optional[Any] = None
    X_test: Optional[Any] = None
    y_test: Optional[Any] = None
    random_forest_evaluation_metrics: Optional[Any] = None
    xgboost_evaluation_metrics: Optional[Any] = None
    feature_columns: Optional[Any] = None
    evaluation_narrative: Optional[str] = None
    selected_model: Optional[Any] = None
    selected_model_name: Optional[str] = None
    selection_reason: Optional[str] = None

    class Config:
        arbitrary_types_allowed = True

# Sanity check
test_state = ModelAgentState()
print("ModelAgentState fields:", list(test_state.dict().keys()))

ModelAgentState fields: ['messages', 'raw_df', 'processed_df', 'random_forest_model', 'xgboost_model', 'X_train', 'y_train', 'X_test', 'y_test', 'random_forest_evaluation_metrics', 'xgboost_evaluation_metrics', 'feature_columns', 'evaluation_narrative', 'selected_model', 'selected_model_name', 'selection_reason']


/tmp/ipykernel_4039634/2813458658.py:6: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class ModelAgentState(BaseModel):
/tmp/ipykernel_4039634/2813458658.py:32: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print("ModelAgentState fields:", list(test_state.dict().keys()))


In [10]:
class PredictionAgentState(BaseModel):
    # Pipeline log
    messages: list[str] = Field(default_factory=list)

    # Required fields
    applied_model: Optional[Any] = None
    jd: Optional[dict] = None
    prediction: Optional[float] = None
    prediction_narrative: Optional[str] = None
    prediction_reasoning: Optional[str] = None

    class Config:
        arbitrary_types_allowed = True

# Sanity check
test_state = PredictionAgentState()
print("PredictionAgentState fields:", list(test_state.dict().keys()))

PredictionAgentState fields: ['messages', 'applied_model', 'jd', 'prediction', 'prediction_narrative', 'prediction_reasoning']


/tmp/ipykernel_4039634/1497656030.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class PredictionAgentState(BaseModel):
/tmp/ipykernel_4039634/1497656030.py:17: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print("PredictionAgentState fields:", list(test_state.dict().keys()))


# Add SKILL.md

In [11]:
# Create folders for each SKILL.md
os.makedirs("skills/preprocess", exist_ok=True)
os.makedirs("skills/random_forest_train", exist_ok=True)
os.makedirs("skills/xgboost_train", exist_ok=True)
os.makedirs("skills/random_forest_eval", exist_ok=True)
os.makedirs("skills/xgboost_eval", exist_ok=True)
os.makedirs("skills/model_selection", exist_ok=True)
os.makedirs("skills/run_inference", exist_ok=True)
os.makedirs("skills/generate_recommendation", exist_ok=True)

In [12]:
RANDOM_FOREST_TRAIN = """---
name: random_forest_train
description: Train a Random Forest Regressor on the preprocessed AI & Data Jobs salary dataset using a fixed train/test split. Deterministic Python, no LLM involved.
tags: [training, random forest, regression, sklearn]
mode: organisational
---

# Model 1 (Random Forest) Training Skill

## Role
This is an organisational skill. Model fitting with fixed hyperparameters is deterministic. The LLM is not involved. The Python code in random_forest_train_agent was written to satisfy this specification.

## When to use
Invoke this skill immediately after preprocess_agent has written X_train, y_train, and feature_columns to agent state. This skill must run before random_forest_eval_agent.
It runs in parallel with xgboost_train_agent and has no dependency on it.

## Specification
1. Read X_train, y_train, and feature_columns from state.
2. Instantiate RandomForestRegressor(n_estimators=200, min_samples_leaf=4,
   random_state=42, n_jobs=-1).
3. Fit the model on X_train and y_train.
4. Store the fitted model in state.random_forest_model.

## Inputs
- X_train: pd.DataFrame, one-hot encoded training features from preprocess_agent
- y_train: pd.Series, continuous salary target values for the training split
- feature_columns: list[str], ordered list of feature names after encoding

## Outputs
- random_forest_model: fitted RandomForestRegressor object

## Output format
random_forest_model is a scikit-learn fitted estimator object, ready to be consumed directly by random_forest_eval_agent via state. The field name itself identifies the model type for downstream agents and the
Gradio interface.

## Business context
This stage produces the first candidate model that will ultimately predict salary ranges for AI and data roles. The output feeds directly into the evaluation and selection stages, which together determine which model
a hiring manager or job seeker sees results from in the final interface.

## Critical Rule
Do not perform any feature engineering or encoding here. All transformations must have been completed by preprocess_agent before this skill runs.
"""

with open("skills/random_forest_train/SKILL.md", "w") as f:
    f.write(RANDOM_FOREST_TRAIN.strip())

print("Written: skills/random_forest_train/SKILL.md")

Written: skills/random_forest_train/SKILL.md


In [13]:
XGBOOST_TRAIN = """---
name: xgboost_train
description: Train an XGBoost Regressor on the preprocessed AI & Data Jobs salary dataset using a fixed train/test split. Deterministic Python, no LLM involved.
tags: [training, xgboost, regression, gradient boosting]
mode: organisational
---

# Model 2 (XGBoost) Training Skill

## Role
This is an organisational skill. Model fitting with fixed hyperparameters is deterministic. The LLM is not involved. The Python code in xgboost_train_agent was written to satisfy this specification.

## When to use
Invoke this skill immediately after preprocess_agent has written X_train, y_train, and feature_columns to agent state. This skill must run before xgboost_eval_agent.
It runs in parallel with random_forest_train_agent and has no dependency on it.

## Specification
1. Read X_train, y_train, and feature_columns from state.
2. Instantiate XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
   subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist").
3. Fit the model on X_train and y_train.
4. Store the fitted model in state.xgboost_model.

## Inputs
- X_train: pd.DataFrame, one-hot encoded training features from preprocess_agent
- y_train: pd.Series, continuous salary target values for the training split
- feature_columns: list[str], ordered list of feature names after encoding

## Outputs
- xgboost_model: fitted XGBRegressor object

## Output format
xgboost_model is an XGBoost fitted estimator object, ready to be consumed directly by xgboost_eval_agent via state. XGBoost accepts pd.DataFrame inputs natively; no numpy conversion is needed before passing X_train.

## Business context
This stage produces the second candidate model for predicting salary ranges for AI and data roles. Gradient boosting is included as a comparison against Random Forest because it often captures complex non-linear feature
interactions in structured tabular data, which is relevant here given the mix of categorical job attributes driving salary variation.

## Critical Rule
Do not perform any feature engineering or encoding here. All transformations must have been completed by preprocess_agent before this skill runs.
"""

with open("skills/xgboost_train/SKILL.md", "w") as f:
    f.write(XGBOOST_TRAIN.strip())

print("Written: skills/xgboost_train/SKILL.md")

Written: skills/xgboost_train/SKILL.md


**Note I will improve the evaluation skill.md later.**

In [ ]:
from sklearn.model_selection import train_test_split

PREPROCESS = """---
name: preprocess
description: Prepare the AI & Data Jobs salary dataset for model training with deterministic cleaning, one-hot encoding, and a fixed train/test split.
tags: [preprocessing, data ingestion, train-test split, encoding]
mode: organisational
---

# Preprocess Skill

## When to use
Invoke this skill immediately after ingestion_agent has loaded the raw dataframe into state.raw_df.

## Specification
1. Read the raw dataframe from state.raw_df.
2. Separate salary_usd as the regression target.
3. Drop identifier columns that should not be used for training.
4. Fill missing categorical values with 'Unknown' and numeric values with the median.
5. One-hot encode categorical columns using pandas.
6. Create a deterministic train/test split with random_state=42.
7. Store processed_df, X_train, X_test, y_train, y_test, and feature_columns back into ModelAgentState.
"""

with open("skills/preprocess/SKILL.md", "w") as f:
    f.write(PREPROCESS.strip())

print("Written: skills/preprocess/SKILL.md")

def preprocess_agent(state: ModelAgentState) -> ModelAgentState:
    print("[preprocess_agent] Starting preprocessing...")

    if state.raw_df is None or len(state.raw_df) == 0:
        raise ValueError("state.raw_df must contain the ingested dataset before preprocessing.")

    df = state.raw_df.copy()
    target_column = "salary_usd"

    if target_column not in df.columns:
        raise ValueError(f"Expected '{target_column}' to exist in the raw dataframe.")

    y = df[target_column].copy()
    X = df.drop(columns=[target_column], errors="ignore").copy()
    X = X.drop(columns=["id"], errors="ignore")

    categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_columns = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

    if categorical_columns:
        X[categorical_columns] = X[categorical_columns].fillna("Unknown")

    if numeric_columns:
        X[numeric_columns] = X[numeric_columns].apply(lambda col: col.fillna(col.median()))

    X_processed = pd.get_dummies(X, columns=categorical_columns, drop_first=False)

    X_train, X_test, y_train, y_test = train_test_split(
        X_processed,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
    )

    state.processed_df = pd.concat([X_processed, y.rename(target_column)], axis=1)
    state.X_train = X_train
    state.y_train = y_train
    state.X_test = X_test
    state.y_test = y_test
    state.feature_columns = X_processed.columns.tolist()
    state.messages.append(
        f"[preprocess_agent] Prepared {len(df)} rows into {X_processed.shape[1]} features with {len(X_train)} train and {len(X_test)} test samples."
    )

    print(
        f"[preprocess_agent] Complete. Rows: {len(df)}, Features: {X_processed.shape[1]}, "
        f"Train: {len(X_train)}, Test: {len(X_test)}"
    )

    return state

In [14]:
# Create random_forest_evaluation SKILL.md
RANDOM_FOREST_EVAL = """---

name: random_forest_evaluation_agent
description: Evaluate the Random Forest model performance using regression metrics (RMSE, MAE, and R²) and visualizations.
mode: LLM-assisted
tags: [evaluation, random forest, regression
---

## When to use
Execute after random_forest_train_agent has completed training.

## How to execute
1. Load trained model from state.random_forest_model
2. Predict on the held-out test set (X_test)
3. Calculate the metrics:
   - RMSE: measure the average deviation between predicted and actual values.
   - MAE: measure the average absolute difference between predicted and actual values.
   - R²: measure the proportion of variance in the target variable that is explained by the model.
4. Generate scatter plot (predicted vs actual)
5. Generate residual plot showing errors acrocss salary range
6. Use GPT-4o-mini to interpret metrics in plain English for non-technical stakeholders (HR)
7. Store the evaluation metrics in state.random_forest_evaluation_metrics

## Inputs from agent state
- random_forest_model: Trained model
- X_test: Test features
- y_test: True test salaries

## Outputs to agent state
- random_forest_evaluation_metrics: dict with 'rmse', 'mae', 'r2'
- Messages: Plain English interpretation

## Output format
Metrics example:
{
    'rmse': 15234.56,
    'mae': 12045.23,
    'r2': 0.7623
}

## Notes
- Use the TEST set only, do not evaluate the train data
- GPT-4o-mini promot: "Explain what RMSE, MAE, and R² mean in the context of salary prediction accuracy. Keep in mind that the audiance is an HR"


"""

with open("skills/random_forest_eval/SKILL.md", "w") as f:
    f.write(RANDOM_FOREST_EVAL.strip())
print("Written: skills/random_forest_eval/SKILL.md")

Written: skills/random_forest_eval/SKILL.md


In [15]:
# Create xgboost_eval SKILL.md
XGBOOST_EVAL = """---

name: xgboost_evaluation_model
description: Evaluate XGBoost model using RMSE, MAE, R² and visualizations
---

# XGBoost Evaluation Skill

## When to use
Execute after xgboost_train_agent has completed training.

## How to execute
It is the similar process to the random forest evaluation, but for XGBoost.

1. Load trained model from state.xgboost_model
2. Predict on the hold-out test set X_test (NOT the training set)
3. Calculate RMSE, MAE, R²
4. Generate scatter plot (predicted vs actual)
5. Generate residual plot
6. Use GPT-4o-mini to interpret metrics in plain English that HR can easily undertand
7. Store the evaluation metrics in state.xgboost_evaluation_metrics

## Inputs from agent state
- xgboost_model: Trained model
- X_test: Test features
- y_test: True test salaries

## Outputs to agent state
- xgboost_evaluation_metrics: dictionary with 'rmse', 'mae', 'r2'
- Messages: Plain English interpretation from GPT
- Update state message with evaluation summary

## Output format
Metrics example:
{
    'rmse': 1234.56,
    'mae': 1234.56,
    'r2': 0.1234
}
"""

with open("skills/xgboost_eval/SKILL.md", "w") as f:
    f.write(XGBOOST_EVAL.strip())
print("Written: skills/xgboost_eval/SKILL.md")

Written: skills/xgboost_eval/SKILL.md


In [16]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def random_forest_eval_agent(state: ModelAgentState) -> ModelAgentState:
    """Evaluate Random Forest model on test set"""
    print("[random_forest_eval_agent] Starting evaluation...")

    # 1. Get predictions
    y_pred = state.random_forest_model.predict(state.X_test)

    # 2. Calculate metrics
    rmse = np.sqrt(mean_squared_error(state.y_test, y_pred))
    mae = mean_absolute_error(state.y_test, y_pred)
    r2 = r2_score(state.y_test, y_pred)

    metrics = {
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }

    # 3. Use GPT to interpret
    prompt = f"""Interpret these Random Forest regression metrics for an HR manager:
- RMSE: ${rmse:,.2f}
- MAE: ${mae:,.2f}
- R²: {r2:.4f}

Context: Predicting AI salaries ranging $30k-$200k.
Write 3-4 sentences in plain English explaining what these numbers mean for salary prediction accuracy."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You translate ML metrics into plain English for HR managers."},
            {"role": "user", "content": prompt}
        ]
    )

    interpretation = response.choices[0].message.content

    # 4. Store results
    state.random_forest_evaluation_metrics = metrics
    state.messages.append(f"[random_forest_eval_agent] RMSE=${rmse:,.0f}, MAE=${mae:,.0f}, R²={r2:.3f}")
    state.messages.append(f"Interpretation: {interpretation}")

    print(f"[random_forest_eval_agent] Complete.")
    print(f"  RMSE: ${rmse:,.0f}")
    print(f"  MAE:  ${mae:,.0f}")
    print(f"  R²:   {r2:.3f}")
    print(f"\n{interpretation}\n")

    return state


In [17]:
def xgboost_eval_agent(state: ModelAgentState) -> ModelAgentState:
    """Evaluate XGBoost model on test set"""
    print("[xgboost_eval_agent] Starting evaluation...")

    # 1. Get predictions
    y_pred = state.xgboost_model.predict(state.X_test)

    # 2. Calculate metrics
    rmse = np.sqrt(mean_squared_error(state.y_test, y_pred))
    mae = mean_absolute_error(state.y_test, y_pred)
    r2 = r2_score(state.y_test, y_pred)

    metrics = {
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }

    # 3. Use GPT to interpret
    prompt = f"""Interpret these XGBoost regression metrics for an HR manager:
- RMSE: ${rmse:,.2f}
- MAE: ${mae:,.2f}
- R²: {r2:.4f}

Context: Predicting AI salaries ranging $30k-$200k.
Write 3-4 sentences in plain English explaining what these numbers mean for salary prediction accuracy."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You translate ML metrics into plain English for HR managers."},
            {"role": "user", "content": prompt}
        ]
    )

    interpretation = response.choices[0].message.content

    # 4. Store results
    state.xgboost_evaluation_metrics = metrics
    state.messages.append(f"[xgboost_eval_agent] RMSE=${rmse:,.0f}, MAE=${mae:,.0f}, R²={r2:.3f}")
    state.messages.append(f"Interpretation: {interpretation}")

    print(f"[xgboost_eval_agent] Complete.")
    print(f"  RMSE: ${rmse:,.0f}")
    print(f"  MAE:  ${mae:,.0f}")
    print(f"  R²:   {r2:.3f}")
    print(f"\n{interpretation}\n")

    return state

In [18]:
# Create model_selection/SKILL.md
MODEL_SELECTION = """---
name: model_selection
description: Compare the results of two models and select the better one and give the reason
mode: LLM-driven
---

# Model Selection Skill

## When to use
- User asks to compare two models
- User wonders which model performs better in prediction
- User wants to choose a better model from candidate models

## How to execute
- Receive the evaluation metrics (RMSE, MAE, and R2) of the two models from ModelAgentState
- Compare the metrics and select the model with a lower RMSE, a lower MAE, and a higher R2
- If the three metrics have conflicts, choose the model with a lower RMSE
- Store the selected model and reason as selected_model and selection_reason in ModelAgentState
- Tell the user which model is better and provide its evaluation metrics

## Input from agent state
- random_forest_evaluation_metrics: dict
- xgboost_evaluation_metrics: dict

## Output to agent state
Store the selected model and reason as selected_model and selection_reason in ModelAgentState

## Output format
Always return output in json format {"selected_model": selected_model, "model_name": model_name, "selection_reason": reasoning}
- selected_model is either random_forest_model or xgboost_model
- model_name is either "Random Forest" or "XGBoost"
- reasoning is a string starting "[model_name] performs better because ...", followed by the comparison of evaluation metrics, such as a lower RMSE, or a higher R2

## Constraints
- reasoning should not exceed 150 words total
- Use plain, accessible language
- Get the evaluation metrics from evaluation agents. Do not use fake numbers
"""

with open("skills/model_selection/SKILL.md", "w") as f:
    f.write(MODEL_SELECTION.strip())

print("Written: skills/model_selection/SKILL.md")

Written: skills/model_selection/SKILL.md


In [19]:
# Create run_inference/SKILL.md
RUN_INFERENCE = """---
name: run_inference
description: Predict the salary for a given job title, experience level and educational background, using the chosen model by model selection agent.
Use when the user inputs the job requirements and expects a estimated salary
mode: LLM-driven
---

# Run Inference Skill

## When to use
- User wants an estimated salary for a given job title, experience level and educational background
- User requests a recommended salary for a data or AI related job posting
- User wants to know the market rate for hiring a person in a given position

## How to execute
- LLM reads user prompt which contains key features
- Get selected_model from ModelAgentState
- Use the given features to predict the salary using the selected_model

## Input from agent state
- Store the key features into PredictionAgentState
- Get selected_model from models/selected_model.pkl

## Output to agent state
- Store the predicted salary as prediction in PredictionAgentState

## Output format
Always structure your response like this:
- Respond with "Recommended salary for [job title] with [experience level] years of experience and [educational background] is $[salary range]."
- The salary range is calculated by the model prediction with a 10% upper and lower margin

## Constraints
- Use plain, accessible language
- Predicted salary should be from the chosen model
- Never provide fake numbers
"""

with open("skills/run_inference/SKILL.md", "w") as f:
    f.write(RUN_INFERENCE.strip())

print("Written: skills/run_inference/SKILL.md")

Written: skills/run_inference/SKILL.md


In [20]:
# Load skill body of SKILL.md
def load_skill_body(filepath: str) -> str:
    with open(filepath, "r") as f:
        content = f.read()

    parts = content.split("---", maxsplit=2)
    if len(parts) < 3:
        return content  # No front matter found, return everything.

    return parts[2].strip()

In [21]:
load_skill_body("skills/model_selection/SKILL.md")

'# Model Selection Skill\n\n## When to use\n- User asks to compare two models\n- User wonders which model performs better in prediction\n- User wants to choose a better model from candidate models\n\n## How to execute\n- Receive the evaluation metrics (RMSE, MAE, and R2) of the two models from ModelAgentState\n- Compare the metrics and select the model with a lower RMSE, a lower MAE, and a higher R2\n- If the three metrics have conflicts, choose the model with a lower RMSE\n- Store the selected model and reason as selected_model and selection_reason in ModelAgentState\n- Tell the user which model is better and provide its evaluation metrics\n\n## Input from agent state\n- random_forest_evaluation_metrics: dict\n- xgboost_evaluation_metrics: dict\n\n## Output to agent state\nStore the selected model and reason as selected_model and selection_reason in ModelAgentState\n\n## Output format\nAlways return output in json format {"selected_model": selected_model, "model_name": model_name, "se

# Define agents in ModelAgentState

In [22]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def random_forest_train_agent(state: ModelAgentState) -> ModelAgentState:
    skill_name = "random_forest_train"
    skill_path = "skills/random_forest_train/SKILL.md"

    with open(skill_path, "r") as f:
        skill_content = f.read()
    print(f"[random_forest_train_agent] Skill: '{skill_name}' | Mode: organisational")
    # Mode is organisational: run deterministic Python, no LLM call

    model = RandomForestRegressor(
        n_estimators     = 200,
        min_samples_leaf = 4,
        random_state     = 42,
        n_jobs           = -1,
    )
    model.fit(state.X_train, state.y_train)
    print(f"[random_forest_train_agent] Training complete. "
          f"Rows: {state.X_train.shape[0]}, Features: {state.X_train.shape[1]}")

    state.random_forest_model = model
    state.messages.append("[random_forest_train_agent] Random Forest trained successfully.")
    return state


def xgboost_train_agent(state: ModelAgentState) -> ModelAgentState:
    skill_name = "xgboost_train"
    skill_path = "skills/xgboost_train/SKILL.md"

    with open(skill_path, "r") as f:
        skill_content = f.read()
    print(f"[xgboost_train_agent] Skill: '{skill_name}' | Mode: organisational")
    # Mode is organisational: run deterministic Python, no LLM call

    model = XGBRegressor(
        n_estimators     = 200,
        learning_rate    = 0.05,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        random_state     = 42,
        tree_method      = "hist",
    )
    model.fit(state.X_train, state.y_train)
    print(f"[xgboost_train_agent] Training complete. "
          f"Rows: {state.X_train.shape[0]}, Features: {state.X_train.shape[1]}")

    state.xgboost_model = model
    state.messages.append("[xgboost_train_agent] XGBoost trained successfully.")
    return state

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

client = OpenAI()

def model_selection_agent(state: ModelAgentState) -> ModelAgentState:
    """Get evaluation metrics from two evaluation agents"""
    random_forest_metrics_text = None
    xgboost_metrics_text = None

    for metric in state.random_forest_evaluation_metrics.keys():
        value = state.random_forest_evaluation_metrics[metric]
        random_forest_metrics_text.append(f"{metric}: {round(value, 2)}\n")

    for metric in state.xgboost_evaluation_metrics.keys():
        value = state.xgboost_evaluation_metrics[metric]
        xgboost_metrics_text.append(f"{metric}: {round(value, 2)}\n")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        message=[
            {"role": "system", "content": (load_skill_body("skills/model_selection/SKILL.md"))},
            {"role": "user", "content": (
             f"Random forest evaluation metrics: {random_forest_metrics_text}"
             f"XGBoost evaluation metrics: {xgboost_metrics_text}"
             )
            }
        ]
    )

    decision = json.loads(response.choices[0].message.content)
    state.selected_model = decision['selected_model']
    state.selected_model_name = decision['model_name']
    state.selection_reason = decision['selection_reason']
    state.messages.append(f"[model_selection_agent] Agent chose {decision['model_name']}. Reason: {decision['selection_reason']}.")
    return state


In [ ]:
# Build LangGraph for ModelAgentState
from langgraph.graph import StateGraph, END

graph_model = StateGraph(ModelAgentState)
graph_model.add_node("preprocess", preprocess_agent)
graph_model.add_node("random_forest_train", random_forest_train_agent)
graph_model.add_node("xgboost_train", xgboost_train_agent)
graph_model.add_node("random_forest_eval", random_forest_eval_agent)
graph_model.add_node("xgboost_eval", xgboost_eval_agent)
graph_model.add_node("model_selection", model_selection_agent)

graph_model.set_entry_point("preprocess")
graph_model.add_edge("preprocess", "random_forest_train")
graph_model.add_edge("preprocess", "xgboost_train")
graph_model.add_edge("random_forest_train", "random_forest_eval")
graph_model.add_edge("xgboost_train", "xgboost_eval")
graph_model.add_edge("random_forest_eval", "model_selection")
graph_model.add_edge("xgboost_eval", "model_selection")
graph_model.add_edge("model_selection", END)

app_model = graph_model.compile()
# result_model = app_model.invoke(ModelAgentState())

NameError: name 'preprocess_agent' is not defined

In [ ]:
import pickle

with open("models/selected_model.pkl", "wb") as f:
    pickle.dump(ModelAgentState.selected_model, f)

In [ ]:
import pickle

def load_best_model():
    with open("models/selected_model.pkl", "rb") as f:
        model = pickle.load(f)
    return model

In [ ]:
def load_best_model_node(state):
    state["model"] = load_best_model()
    return state

In [ ]:
client = OpenAI()

def run_inference_agent(state: PredictionAgentState) -> ModelAgentState:


# Gradio

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 💼 AI Talent Salary Prediction System")
    gr.Markdown("Get data-driven salary recommendations for AI professionals")

    with gr.Row():
        with gr.Column():
            job_role = gr.Dropdown(
                choices=['Machine Learning Engineer', 'Data Scientist', 'AI Engineer',
                        'NLP Engineer', 'Computer Vision Engineer', 'Research Scientist'],
                label="Job Role"
            )
            ai_spec = gr.Dropdown(
                choices=['NLP', 'Computer Vision', 'LLM', 'MLOps', 'Reinforcement Learning',
                        'Generative AI', 'Analytics', 'Forecasting'],
                label="AI Specialization"
            )
            exp_level = gr.Dropdown(
                choices=['Entry', 'Mid', 'Senior', 'Lead'],
                label="Experience Level"
            )
            exp_years = gr.Slider(0, 20, value=5, step=1, label="Years of Experience")
            education = gr.Dropdown(
                choices=['Bootcamp', 'Bachelor', 'Master', 'PhD', 'Diploma'],
                label="Education"
            )

        with gr.Column():
            country = gr.Dropdown(
                choices=['USA', 'Singapore', 'India', 'UK', 'Germany', 'Canada',
                        'Australia', 'France', 'Japan', 'Netherlands', 'Brazil', 'UAE'],
                label="Country"
            )
            industry = gr.Dropdown(
                choices=['Tech', 'Finance', 'Healthcare', 'Retail', 'Consulting',
                        'Automotive', 'Education', 'Energy', 'Telecom', 'Gaming'],
                label="Industry"
            )
            company_size = gr.Dropdown(
                choices=['Startup', 'Small', 'Medium', 'Large', 'Enterprise'],
                label="Company Size"
            )
            work_mode = gr.Dropdown(
                choices=['Remote', 'Hybrid', 'Onsite'],
                label="Work Mode"
            )

    predict_btn = gr.Button("🔮 Predict Salary", variant="primary", size="lg")

    with gr.Row():
        with gr.Column():
            output_text = gr.Markdown(label="Recommendation")
        with gr.Column():
            similar_table = gr.Dataframe(label="Similar Roles in Market")

    predict_btn.click(
        fn=predict_salary,
        inputs=[job_role, ai_spec, exp_level, exp_years, education,
                country, industry, company_size, work_mode],
        outputs=[output_text, similar_table]
    )

demo.launch(share=True)  # share=True required for submission

/tmp/ipykernel_1333/4152057547.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


NameError: name 'predict_salary' is not defined